## Build Manual-Annotation Sample CSV

Data-prep notebook (no model loading) that draws a small random sample of problems from a generation output and formats them into a CSV for manual review.

In [20]:
import random
random.seed(42)

# Pick 5 random numbers from 0 to 255
random_numbers = [random.randint(0, 255) for _ in range(5)]
print(random_numbers)


[57, 12, 140, 125, 114]


### Load generation output and sample problems

In [21]:
import sys
sys.path.append("../../src")

import pandas as pd

import _config

prompt_config = _config.PromptConfig(model_type="GPT-OSS_stepwise", prompt_type="h")
generation_run_config = _config.RunConfig(
    experiment_root="experiments/token_intervention",
    output_filename="generation_h.csv",
)
generation_path = _config.build_run_output_path(prompt_config, generation_run_config)
annotation_path = generation_path.with_name("generation_h_filtered_for_manual_annotation.csv")

prompts_h = _config.load_prompts(prompt_config)
divided_prompts_h = _config.load_divided_prompts(prompt_config)

print(f"Rows in {prompt_config.filename()}: {len(prompts_h)}")
print(f"Rows in {prompt_config.divided_filename()}: {len(divided_prompts_h)}")

Rows in prompts_h.csv: 256
Rows in divided_prompts_h.csv: 3072


In [22]:
import pandas as pd

# Read the CSV file
df = pd.read_csv(generation_path)

# Group rows based on whether they share both 'base_1_num' and 'base_2_num'
grouped = df.groupby(['base_1_num', 'base_2_num'])

# Pick groups with indices specified by random_numbers.
selected_groups = [group for idx, group in enumerate([group for _, group in grouped]) if idx in random_numbers]

print(selected_groups)

[     base_1_digits base_2_digits  base_1_num  base_2_num  base_sum  \
1332     [1, 1, 2]     [1, 5, 6]         112         156       268   
1333     [1, 1, 2]     [1, 5, 6]         112         156       268   
1334     [1, 1, 2]     [1, 5, 6]         112         156       268   
1335     [1, 1, 2]     [1, 5, 6]         112         156       268   
1336     [1, 1, 2]     [1, 5, 6]         112         156       268   
1337     [1, 1, 2]     [1, 5, 6]         112         156       268   
1338     [1, 1, 2]     [1, 5, 6]         112         156       268   
1339     [1, 1, 2]     [1, 5, 6]         112         156       268   
1340     [1, 1, 2]     [1, 5, 6]         112         156       268   
1341     [1, 1, 2]     [1, 5, 6]         112         156       268   
1342     [1, 1, 2]     [1, 5, 6]         112         156       268   
1343     [1, 1, 2]     [1, 5, 6]         112         156       268   

     source_1_digits source_2_digits  source_1_num  source_2_num  source_sum  \
1332    

### Select and format sample rows

Extracts the reasoning-chain text and final answer from the generated output into separate columns for easier manual reading.

### Select and format sample rows (continued)

In [23]:
# Concatenate all groups into a single DataFrame, selecting specific columns of interest
cols = [
    "base_1_num",
    "base_2_num",
    "base_sum",
    "source_1_num",
    "source_2_num",
    "source_sum",
    "intervention_id",
    "base_before",
    "base_number",
    "base_after",
    "source_before",
    "source_number",
    "source_after",
    "counterfactual_sum",
    "generated_text",
]
selected_df = pd.concat(selected_groups)[cols]# .rename(columns={"base_sum": "factual_sum", "base_before": "before_intervention", "source_number": "intervention", "generated_text": "generated_reasoning"})
# selected_df["before_intervention"] = selected_df["before_intervention"].apply(lambda x: x.split("<|channel|>analysis<|message|>")[-1] if isinstance(x, str) else x)
# split_strings = selected_df["generated_reasoning"].apply(
#     lambda x: x.split("<|end|><|start|>assistant<|channel|>final<|message|>") if isinstance(x, str) else [x, None]
# )
# selected_df["generated_reasoning"] = split_strings.apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)
# selected_df["generated_output"] = split_strings.apply(lambda x: x[1].replace("<|return|>","") if isinstance(x, list) and len(x) > 1 else None)

selected_df["base_before"] = selected_df["base_before"].apply(lambda x: x.split("<|channel|>analysis<|message|>")[-1] if isinstance(x, str) else x)
split_strings = selected_df["generated_text"].apply(
    lambda x: x.split("<|end|><|start|>assistant<|channel|>final<|message|>") if isinstance(x, str) else [x, None]
)
selected_df["generated_text"] = split_strings.apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)
selected_df["generated_output"] = split_strings.apply(lambda x: x[1].replace("<|return|>","") if isinstance(x, list) and len(x) > 1 else None)


print(selected_df)

      base_1_num  base_2_num  base_sum  source_1_num  source_2_num  \
1332         112         156       268           112           756   
1333         112         156       268           112           756   
1334         112         156       268           112           756   
1335         112         156       268           112           756   
1336         112         156       268           112           756   
1337         112         156       268           112           756   
1338         112         156       268           112           756   
1339         112         156       268           112           756   
1340         112         156       268           112           756   
1341         112         156       268           112           756   
1342         112         156       268           112           756   
1343         112         156       268           112           756   
708          148         753       901           148           453   
709          148    

### Write output

In [24]:
annotation_path.parent.mkdir(parents=True, exist_ok=True)
selected_df.to_csv(annotation_path, index=False)
